# Fiorell.IA LoRA — Master Runbook (Colab A100 / Google Drive)

Conservative wrapper for the existing training → export → eval → decision workflow.

This notebook is the Colab/Drive-first Fiorell.IA release path.
It does not use managed cloud endpoints, external VMs, or non-Drive artifact stores.

It does **not** change:
- base model (`Qwen/Qwen2.5-3B-Instruct`);
- LoRA method;
- dataset semantics;
- abstention/refusal behavior.

It centralizes path configuration, preflight checks, adapter export, adapter evaluation and final verdict artifacts in Google Drive.


In [ ]:
# 00 - Fiorell.IA Drive-first bootstrap
from pathlib import Path
import urllib.request

BOOTSTRAP_REL = "fiorellia_colab_drive_bootstrap.py"
DRIVE_REPO_ROOT = Path("/content/drive/MyDrive/regulatory-insight-engine")
MAC_DRIVE_REPO_ROOT = Path("/Users/itsgennymac/Library/CloudStorage/GoogleDrive-sfn.gns@gmail.com/Il mio Drive/regulatory-insight-engine")
BOOTSTRAP_URL = "https://raw.githubusercontent.com/TheGenesisAIStory/regulatory-insight-engine/main/fiorellia_colab_drive_bootstrap.py"

bootstrap_path = (DRIVE_REPO_ROOT if Path("/content").exists() else MAC_DRIVE_REPO_ROOT) / BOOTSTRAP_REL
bootstrap_path.parent.mkdir(parents=True, exist_ok=True)
if not bootstrap_path.exists() or "drive_first_bootstrap" not in bootstrap_path.read_text(encoding="utf-8", errors="ignore"):
    bootstrap_path.write_text(urllib.request.urlopen(BOOTSTRAP_URL).read().decode("utf-8"), encoding="utf-8")

exec(bootstrap_path.read_text(encoding="utf-8"), globals())


In [ ]:
# 00_config — edit only this cell
from pathlib import Path

RUN_ENV = "colab"  # colab | local

REPO_ROOT = Path("/content/regulatory-insight-engine") if RUN_ENV == "colab" else Path.cwd()
CONFIG_PATH = REPO_ROOT / "fiorellia/training/configs/config_lora_behavior_20260421.yaml"

TRAIN_SCRIPT = REPO_ROOT / "fiorellia/training/train_lora_behavior_v1.py"
PROMPT_HARNESS_SCRIPT = REPO_ROOT / "fiorellia/eval/prompt_harness.py"

ARTIFACT_ROOT = Path("/content/drive/MyDrive/fiorellia-runs/final_delivery_latest") if RUN_ENV == "colab" else REPO_ROOT / "artifacts/fiorellia"
ADAPTER_ZIP = ARTIFACT_ROOT / "fiorellia_lora_adapter.zip"
METRICS_JSON = ARTIFACT_ROOT / "metrics_summary.json"
FINAL_VERDICT_MD = ARTIFACT_ROOT / "final_verdict.md"
DIAGNOSTICS_JSON = ARTIFACT_ROOT / "eval_diagnostics.json"

SYSTEM_PROMPT_PATH = REPO_ROOT / "fiorellia/prompts/system_prompt.md"
EVAL_SET_PATH = REPO_ROOT / "fiorellia/eval/eval_set.jsonl"
BASELINE_JSONL_PATH = REPO_ROOT / "fiorellia/eval/baseline.jsonl"
EVAL_REPORT_DIR = REPO_ROOT / "fiorellia/eval/reports"
ADAPTER_EVAL_JSONL = EVAL_REPORT_DIR / "adapter_eval.jsonl"
SCORED_EVAL_JSONL = ARTIFACT_ROOT / "adapter_eval_scored.jsonl"

print("RUN_ENV:", RUN_ENV)
print("REPO_ROOT:", REPO_ROOT)
print("ARTIFACT_ROOT:", ARTIFACT_ROOT)


## 00b_colab_setup

Mount Google Drive and clone or update the public GitHub repository. This makes `Runtime -> Restart session and run all` work on a clean Colab A100 runtime.


In [ ]:
# 00b_colab_setup — Drive mount + repo clone/pull
import subprocess, sys

if RUN_ENV == "colab":
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    if not REPO_ROOT.exists():
        subprocess.run([
            "git",
            "clone",
            "https://github.com/TheGenesisAIStory/regulatory-insight-engine.git",
            str(REPO_ROOT),
        ], check=True)
    else:
        subprocess.run(["git", "pull", "origin", "main"], cwd=str(REPO_ROOT), check=True)

    sys.path.insert(0, str(REPO_ROOT))
    print("Repo ready:", REPO_ROOT)
    print("Artifacts:", ARTIFACT_ROOT)
else:
    sys.path.insert(0, str(REPO_ROOT))
    print("Local repo:", REPO_ROOT)


## 01_dependencies

Run once on a clean runtime. Avoid scattered `pip install -U` cells later in the notebook.


In [ ]:
# Colab A100 conservative dependency cell.
# Keep ranges broad enough for Colab A100, narrow enough to avoid common breakage.
%pip install -q \
  "transformers>=4.45,<4.52" \
  "datasets>=2.20,<3.0" \
  "accelerate>=0.33,<1.0" \
  "peft>=0.12,<0.16" \
  "trl>=0.9,<0.13" \
  "bitsandbytes>=0.43,<0.46" \
  "safetensors>=0.4" \
  "pyyaml>=6.0" 


In [ ]:
# 02_preflight
import subprocess, sys
from pathlib import Path

# Defensive bootstrap: this cell can run even if 00b_colab_setup was skipped.
if not (REPO_ROOT / "fiorellia").exists():
    if RUN_ENV == "colab":
        if not Path("/content/drive").exists():
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
        subprocess.run([
            "git",
            "clone",
            "https://github.com/TheGenesisAIStory/regulatory-insight-engine.git",
            str(REPO_ROOT),
        ], check=True)
    else:
        raise RuntimeError(f"Repository root does not contain fiorellia/: {REPO_ROOT}")
elif RUN_ENV == "colab":
    subprocess.run(["git", "pull", "origin", "main"], cwd=str(REPO_ROOT), check=True)

sys.path.insert(0, str(REPO_ROOT))
print("Python import path head:", sys.path[:3])
print("Repo root exists:", REPO_ROOT.exists())
print("Fiorellia package exists:", (REPO_ROOT / "fiorellia").exists())

from fiorellia.training.fiorellia_colab_pipeline import (
    check_cuda,
    load_config,
    validate_config,
    require_file,
)

gpu_info = check_cuda(require_gpu=True)
config = load_config(CONFIG_PATH)
resolved = validate_config(config, REPO_ROOT)

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print("GPU:", gpu_info)
print("Dataset:", resolved["dataset_path"])
print("Output dir:", resolved["output_dir"])
print("Artifact root:", ARTIFACT_ROOT)


## 03_training

This cell delegates to the existing training script. It does not change training semantics.


In [ ]:
# 03_training — keep disabled until preflight is green
import subprocess, sys

require_file(TRAIN_SCRIPT, "training script")
cmd = [sys.executable, str(TRAIN_SCRIPT), "--config", str(CONFIG_PATH)]
print("+", " ".join(cmd))
subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)

print("Training completed")


In [ ]:
# 04_export — validate adapter directory and create zip
from fiorellia.training.fiorellia_colab_pipeline import zip_adapter, validate_adapter_zip

adapter_dir = resolved["output_dir"]
created_zip = zip_adapter(adapter_dir, ADAPTER_ZIP)
validate_adapter_zip(created_zip)

print("Adapter zip:", created_zip)


## 05_eval

The notebook now runs the adapter eval harness directly after validating inputs. Results are written to `fiorellia/eval/reports/adapter_eval.jsonl`, then scored in the final cell.


In [ ]:
# 05_eval_preflight — run before executing existing eval cells/script
from fiorellia.training.fiorellia_colab_pipeline import validate_adapter_zip

validate_adapter_zip(ADAPTER_ZIP)
require_file(SYSTEM_PROMPT_PATH, "system prompt")
require_file(EVAL_SET_PATH, "eval set")
require_file(BASELINE_JSONL_PATH, "baseline JSONL")

print("Eval inputs validated")
print("Adapter zip:", ADAPTER_ZIP)
print("System prompt:", SYSTEM_PROMPT_PATH)
print("Eval set:", EVAL_SET_PATH)
print("Baseline:", BASELINE_JSONL_PATH)


In [ ]:
# 05a_cleanup_previous_artifacts
for pattern in ["metrics_summary*.json", "final_verdict*.md"]:
    for artifact_path in ARTIFACT_ROOT.glob(pattern):
        artifact_path.unlink()
        print("removed:", artifact_path)

print("Previous None-valued verdict artifacts cleaned")


In [ ]:
# 05b_run_prompt_harness — produce real adapter eval JSONL
import subprocess, sys

require_file(PROMPT_HARNESS_SCRIPT, "prompt harness script")
EVAL_REPORT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(PROMPT_HARNESS_SCRIPT),
    "--adapter_zip", str(ADAPTER_ZIP),
    "--eval_set", str(EVAL_SET_PATH),
    "--system_prompt", str(SYSTEM_PROMPT_PATH),
    "--output", str(EVAL_REPORT_DIR),
]
print("+", " ".join(cmd))
subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)
require_file(ADAPTER_EVAL_JSONL, "adapter eval JSONL")

print("Adapter eval JSONL:", ADAPTER_EVAL_JSONL)


In [ ]:
# 06_compare_and_finalize
# Score the real JSONL produced by prompt_harness.py and write standard artifacts.
from fiorellia.training.fiorellia_colab_pipeline import (
    read_jsonl,
    score_eval_rows,
    write_final_verdict,
    write_json,
    write_jsonl,
)

required_metrics = ["in_scope_grounded", "unsupported_abstention", "out_of_scope_refusal"]
thresholds = {
    "in_scope_grounded": 0.80,
    "unsupported_abstention": 0.90,
    "out_of_scope_refusal": 0.95,
}

rows = read_jsonl(ADAPTER_EVAL_JSONL)
scored_rows, raw_metrics = score_eval_rows(rows)
write_jsonl(scored_rows, SCORED_EVAL_JSONL)

metrics = {}
for name in required_metrics:
    value = raw_metrics.get(name)
    if value is None:
        raise RuntimeError(f"Metric {name} is None. Check eval categories and prompt harness output: {ADAPTER_EVAL_JSONL}")
    value = float(value)
    if not 0.0 <= value <= 1.0:
        raise RuntimeError(f"Metric {name} out of range [0,1]: {value}")
    metrics[name] = value

harness_errors = sum(1 for row in rows if row.get("error"))
diagnostics = {
    "adapter_eval_jsonl": str(ADAPTER_EVAL_JSONL),
    "scored_eval_jsonl": str(SCORED_EVAL_JSONL),
    "harness_errors": harness_errors,
    "subset_counts": raw_metrics.get("subset_counts", {}),
    "italian_style": raw_metrics.get("italian_style"),
}

go = harness_errors == 0 and all(metrics[name] >= thresholds[name] for name in required_metrics)
verdict = "GO" if go else "NO-GO"

write_json(metrics, METRICS_JSON)
write_json(diagnostics, DIAGNOSTICS_JSON)
write_final_verdict(FINAL_VERDICT_MD, verdict, metrics)

print("CONCLUSIONE DEFINITIVA")
print("Esito run:", verdict)
print("Metrics:", metrics)
print("Diagnostics:", diagnostics)
print("Metrics JSON:", METRICS_JSON)
print("Diagnostics JSON:", DIAGNOSTICS_JSON)
print("Final verdict:", FINAL_VERDICT_MD)
